<a href="https://colab.research.google.com/github/marianafreitasc/PedalaRio-HackathonAnalytica/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Limpeza e tratamento

In [ ]:
import pandas as pd
from google.colab import files
import difflib
import numpy as np
import re
import unicodedata

df_acidentes = pd.read_csv('acidentes_transito_isp.csv', sep=';')
df_seguranca = pd.read_csv('indicadores_seguranca_isp.csv', sep=';', encoding='latin1')
df_logradouros = pd.read_csv('logradouros_datario.csv')
df_relacao = pd.read_csv('relacao_cisp_isp.csv', sep=';', encoding='latin1')
df_bairros = pd.read_csv('bairros_datario.csv', sep=',')
df_vias = pd.read_csv('rede_cicloviaria_datario.csv')
df_vias_score = pd.read_csv('vias_score.csv')

ANO_CORTE = 2021 # não serão utilizados dados anteriores ao ano de 2021


# >>> Limpeza e tratamento da tabela de rede cicloviaria

# Padronizando capitalização de nomes de via
df_vias['vias'] = df_vias['vias'].str.upper().str.strip()
df_vias['rp'] = df_vias['rp'].str.upper().str.strip()

# Ajustando nomes das colunas
df_vias = df_vias.rename(columns={
    'tipo_infra': 'infra_tipo',
    'fluxo_infr': 'infra_fluxo',
    'SHAPE__Length': 'extensao_metros',
    'area_plane': 'area_planejamento',
    'rp': 'bairro'
})

# Seleção de colunas relevantes
colunas_vias = ['infra_tipo', 'infra_fluxo', 'vias', 'inicio', 'final', 'bairro', 'extensao_metros', 'area_planejamento']
df_vias = df_vias[colunas_vias].copy()


# >>> Limpeza e tratamento da tabela de logradouros

# Padronizando capitalização de nomes de logradouro e bairro
colunas_nomes = ['completo', 'bairro']
df_logradouros[colunas_nomes] = df_logradouros[colunas_nomes].apply(lambda x: x.str.upper())

# Remove nulos e quebras de linha de todas as colunas de texto
df_logradouros = df_logradouros.replace(r'\n', ' ', regex=True).replace(r'\r', ' ', regex=True)
df_logradouros = df_logradouros.dropna(subset=['hierarquia', 'completo', 'velocidade_regulamentada'])
df_logradouros = df_logradouros[df_logradouros['velocidade_regulamentada'] > 0]
df_logradouros = df_logradouros[~df_logradouros['completo'].str.contains('sem nome', case=False, na=False)]

# Consertando tipagem
df_logradouros['velocidade_regulamentada'] = df_logradouros['velocidade_regulamentada'].astype(int)

# Ajuste de nomes das colunas
df_logradouros = df_logradouros.rename(columns={'completo': 'logradouro', 'velocidade_regulamentada': 'velocidade_maxima',})

# Seleção de colunas relevantes
colunas_logradouros = ['logradouro', 'bairro', 'hierarquia', 'velocidade_maxima']
df_logradouros = df_logradouros[colunas_logradouros].copy()

# Criar nova variável nivel_hierarquia que quantifica os tipos de hierarquia
df_logradouros['nivel_hierarquia'] = df_logradouros['hierarquia'].map({
    'Local': 1, 'Coletora': 1.2, 'Arterial secundária': 2,
    'Arterial primária': 2.5, 'Estrutural': 3
}).fillna(1)

# Ordenar por maior hierarquia e maior velocidade
df_logradouros = df_logradouros.sort_values(by=['bairro', 'logradouro', 'nivel_hierarquia', 'velocidade_maxima'], ascending=[True, True, False, False])
# Remover duplicatas de logradouro mantendo aquele com o risco mais alto
df_logradouros = (df_logradouros.drop_duplicates(subset=['bairro', 'logradouro'], keep='first'))


# >>> Limpeza e tratamento da tabela de acidentes

# Padronizando capitalização de nomes de logradouro
df_acidentes['Logradouro'] = df_acidentes['Logradouro'].str.upper().str.strip()

# Aplicando o filtro de ano corte
df_acidentes = df_acidentes[df_acidentes['Ano'] >= ANO_CORTE].copy()

# Cria nova variável de score ponderado dos acidentes
df_acidentes = (df_acidentes.pivot_table(index=['Ano', 'Logradouro'], columns='Delito', values='Total', aggfunc='sum', fill_value=0).reset_index())

# Ajuste de nomes das colunas
df_acidentes = df_acidentes.rename(columns={
    'Ano': 'ano',
    'Logradouro': 'logradouro',
    'Acidente de trânsito': 'acidente',
    'Atropelamento': 'atropelamento',
    'Homicídio culposo - trânsito': 'homicidio_culposo',
    'Lesão corporal culposa - trânsito': 'lesao_corporal'
})
df_acidentes.columns.name = None

# Seleção de colunas relevantes
colunas_acidentes = ['ano', 'logradouro', 'acidente', 'atropelamento', 'homicidio_culposo', 'lesao_corporal']
df_acidentes = df_acidentes[colunas_acidentes].copy()

# >>> Limpeza e tratamento da tabela auxiliar de bairros

# Remover duplicatas
df_bairros = df_bairros.drop_duplicates()

# Ajuste de nomes das colunas
df_bairros = df_bairros.rename(columns={'nome': 'bairro'})

df_bairros = df_bairros[['bairro']]

# Padronizando capitalização de nomes de bairro
df_bairros['bairro'] = df_bairros['bairro'].str.upper().str.strip()


# >>> Limpeza e tratamento da tabela auxiliar de relação de delegacias com bairros

# Padronizando capitalização do nome das colunas
df_relacao.columns = (df_relacao.columns.str.lower().str.strip())

# Aplicando método explode na coluna "Unidade Territorial" para quebrar conjuntos de bairros em um bairro por linha
df_relacao['unidade_limpa'] = (df_relacao['unidade territorial'].str.replace(r'\(parte\)', '', regex=True).str.strip())
df_relacao['lista_bairros'] = (df_relacao['unidade_limpa'].str.replace(' e ', ', ').str.split(', '))
df_relacao = df_relacao.explode('lista_bairros')
df_relacao['bairro'] = df_relacao['lista_bairros'].str.strip()
colunas_relacao = ['risp', 'aisp', 'cisp', 'bairro']
df_relacao = df_relacao[colunas_relacao].copy()


# >>> Limpeza e tratamento da tabela de indicadores de segurança

# Seleção de colunas relevantes
colunas_seguranca = ['ano', 'risp', 'aisp', 'cisp', 'roubo_bicicleta', 'furto_bicicleta']
df_seguranca = df_seguranca[colunas_seguranca].copy()

# Filtro de ano corte
df_seguranca = df_seguranca[df_seguranca['ano'] >= ANO_CORTE]

# Consertando tipagem
df_seguranca['roubo_bicicleta'] = df_seguranca['roubo_bicicleta'].astype(int)
df_seguranca['furto_bicicleta'] = df_seguranca['furto_bicicleta'].astype(int)


# Aplicando merge que adiciona a coluna de bairro a tabela de seguranca usando tabela auxiliar
df_seguranca = df_seguranca.merge(
    df_relacao, on=['risp', 'aisp', 'cisp'], how='left'
)

# Padronizando capitalização de nomes de bairro
df_seguranca['bairro'] = (df_seguranca['bairro'].str.strip().str.upper())

# Agrupando por bairro e ano para somar os acidentes
df_seguranca = df_seguranca.groupby(['ano', 'bairro']).agg({
    'roubo_bicicleta': 'sum',
    'furto_bicicleta': 'sum'
}).reset_index()


# Filtrar o df_seguranca só com bairros do município do rio usando tabela auxiliar
df_seguranca = df_seguranca.merge(df_bairros, on='bairro', how='inner')

# Criar coluna do total de roubos
colunas_soma_roubos = ['roubo_bicicleta', 'furto_bicicleta']
df_seguranca['total_crimes'] = df_seguranca[colunas_soma_roubos].sum(axis=1)

# Mapear bairros para os logradouros na tabela de acidentes usando a tabela de logradouros
def padronizar_rua(texto):
    if pd.isna(texto):
        return texto
    texto = str(texto).upper().strip()

    substituicoes = {
        'AVENIDA ': 'AV ',
        'AV. ': 'AV ',
        'RUA ': 'R ',
        'R. ': 'R ',
        'GOVERNADOR ': '',
        'PRESIDENTE ': '',
        'MINISTRO ': '',
        ' DO ': ' ',
        ' DA ': ' ',
        ' DE ': ' ',
        ' DOS ': ' ',
        ' DAS ': ' '
    }

    for chave, valor in substituicoes.items():
        texto = texto.replace(chave, valor)

    texto = " ".join(texto.split())
    return texto

# Cria colunas temporarias do logradouro padronizado
df_acidentes['logradouro_padrao'] = df_acidentes['logradouro'].apply(padronizar_rua)
df_logradouros['logradouro_padrao'] = df_logradouros['logradouro'].apply(padronizar_rua)

# Fazemos o merge inicial pela coluna padronizada para adicionar o bairro
df_acidentes = df_acidentes.merge(
    df_logradouros[['logradouro_padrao', 'bairro']].drop_duplicates(subset=['logradouro_padrao']), on='logradouro_padrao', how='left'
)

# Identificando os logradouros que não tem bairro ainda
logradouros_sem_bairro = df_acidentes['bairro'].isna()
sem_bairro_df = df_acidentes[logradouros_sem_bairro].copy()

# Resolvendo logradouros sem bairro identificados
if not sem_bairro_df.empty:
    logradouros_validos = df_logradouros['logradouro_padrao'].unique().tolist()

    def busca_aproximada(rua, escolhas):
      # n=1 com cutoff=0.8
      matches = difflib.get_close_matches(rua, escolhas, n=1, cutoff=0.8)
      return matches[0] if matches else None

    # Aplica a busca aproximada nos logradouros sem bairro
    dicionario_ajuste = {rua: busca_aproximada(rua, logradouros_validos) for rua in sem_bairro_df['logradouro_padrao'].unique()}

    # Atualiza logradouro_padrao para guardar o mais próximo achado
    adjusted_logradouro_padrao_map = {k:v for k,v in dicionario_ajuste.items() if v is not None}
    df_acidentes.loc[logradouros_sem_bairro, 'logradouro_padrao'] = df_acidentes.loc[logradouros_sem_bairro, 'logradouro_padrao'].map(adjusted_logradouro_padrao_map)
    bairro_map_for_corrected_logras = df_logradouros.drop_duplicates(subset=['logradouro_padrao']).set_index('logradouro_padrao')['bairro']
    df_acidentes.loc[logradouros_sem_bairro, 'bairro'] = df_acidentes.loc[logradouros_sem_bairro, 'logradouro_padrao'].map(bairro_map_for_corrected_logras)

# Descarta as colunas padrão
df_acidentes = df_acidentes.drop(columns=['logradouro_padrao'])
df_logradouros = df_logradouros.drop(columns=['logradouro_padrao'])

df_acidentes = df_acidentes.dropna(subset=['bairro'])

# Criar coluna do total de acidentes
colunas_soma_acidentes = ['acidente', 'atropelamento', 'homicidio_culposo', 'lesao_corporal']
df_acidentes['total_delitos'] = df_acidentes[colunas_soma_acidentes].sum(axis=1)

nova_ordem = ['ano', 'logradouro', 'bairro', 'acidente', 'atropelamento', 'homicidio_culposo', 'lesao_corporal', 'total_delitos']
df_acidentes = df_acidentes[nova_ordem]


# >>> Cálculos base para o Score de Risco

df_acidentes['score_d'] = (
    1  * df_acidentes['acidente'] +
    5  * df_acidentes['atropelamento'] +
    10 * df_acidentes['homicidio_culposo'] +
    3  * df_acidentes['lesao_corporal']
)

df_seguranca['score_c'] = (4 * df_seguranca['roubo_bicicleta'] + 2 * df_seguranca['furto_bicicleta'])

# Limites de velocidade e os pesos correspondentes
bins = [0, 39, 49, 59, 69, 120]
labels = [1, 1.2, 1.5, 2, 3]

# Aplicando pesos e tratando os nulos com .fillna(1)
df_logradouros['peso_v'] = pd.cut(df_logradouros['velocidade_maxima'], bins=bins, labels=labels).astype(float).fillna(1)

df_logradouros = df_logradouros.rename(columns={'nivel_hierarquia': 'peso_h'})

# Desconsiderando 2026 (dados incompletos)
df_seguranca = (
    df_seguranca[df_seguranca['ano'] < 2026]
)

# Fazendo a média do score_c dos anos restantes
df_seguranca = (df_seguranca.groupby('bairro', as_index=False).agg({'score_c': 'mean'}))

df_seguranca['score_c'] = (df_seguranca['score_c'].round(1))

# Fazendo a média do score_d dos logradouros por bairro
df_acidentes = (df_acidentes.groupby('bairro', as_index=False).agg({'score_d': 'mean'}))

df_acidentes['score_d'] = (df_acidentes['score_d'].round(1))


# >>> Merges finais para gerar a tabela com todos os dados relevantes e o score calculado

df_final = df_vias.merge(df_seguranca[['bairro', 'score_c']], on='bairro', how='left')

df_final = df_final.merge(df_acidentes[['bairro', 'score_d']], on='bairro', how='left')

df_final['score_c'] = (df_final['score_c'].fillna(0))
df_final['score_d'] = (df_final['score_d'].fillna(0))


# Fazendo merge para consolidar tabela de vias com colunas do logradouro
df_final['via_padrao'] = df_final['vias'].apply(padronizar_rua)
df_logradouros['logradouro_padrao'] = df_logradouros['logradouro'].apply(padronizar_rua)

df_merge = df_final.merge(
    df_logradouros[['logradouro_padrao', 'hierarquia', 'velocidade_maxima', 'peso_v', 'peso_h']].drop_duplicates('logradouro_padrao'),
    left_on='via_padrao', right_on='logradouro_padrao', how='left'
)

# Vias sem correspondente
vias_sem_corresp = df_merge['peso_v'].isna()
logradouros_validos = df_logradouros['logradouro_padrao'].dropna().unique().tolist()

# Criar dicionário apenas para os itens sem match
dicionario_ajuste = {rua: matches[0] for rua in df_merge.loc[vias_sem_corresp, 'via_padrao'].unique()
                    if (matches := difflib.get_close_matches(rua, logradouros_validos, n=1, cutoff=0.85))}

# Procurar aproximados
df_merge.loc[vias_sem_corresp, 'via_padrao'] = df_merge.loc[vias_sem_corresp, 'via_padrao'].map(dicionario_ajuste).fillna(df_merge.loc[vias_sem_corresp, 'via_padrao'])

mapa_v = df_logradouros.drop_duplicates('logradouro_padrao').set_index('logradouro_padrao')['peso_v']
mapa_h = df_logradouros.drop_duplicates('logradouro_padrao').set_index('logradouro_padrao')['peso_h']

df_merge.loc[vias_sem_corresp, 'peso_v'] = df_merge.loc[vias_sem_corresp, 'via_padrao'].map(mapa_v)
df_merge.loc[vias_sem_corresp, 'peso_h'] = df_merge.loc[vias_sem_corresp, 'via_padrao'].map(mapa_h)

# Merge com tabela que tem o score de segurança de infraestrutura das vias calculado
df_final = df_merge.drop(columns=['via_padrao', 'logradouro_padrao'], errors='ignore').copy()
df_vias_score['vias'] = df_vias_score['vias'].str.upper()
df_final = df_final.merge(df_vias_score[['vias', 'safety_score_final_v2']].drop_duplicates('vias'), on='vias', how='left')

# Transformando o score de segurança em risco de infraestrura
df_final[['peso_v', 'peso_h']] = df_final[['peso_v', 'peso_h']].fillna(1)
df_final['risco_infraestrutura'] = 10 - df_final['safety_score_final_v2']

# Aplicando a fórmula do score geral de risco
alpha = 0.3 # peso do score de risco de infraestrura
tem_score = (df_final['score_c'].fillna(0) != 0) | (df_final['score_d'].fillna(0) != 0)
val_crimes = (df_final['score_c'].fillna(0) + df_final['score_d'].fillna(0)) * df_final['peso_v'] * df_final['peso_h']
val_base = df_final['peso_v'] * df_final['peso_h']

df_final['score_risco'] = np.where(tem_score, val_crimes + df_final['risco_infraestrutura'], val_base + alpha * df_final['risco_infraestrutura'])
df_final['score_risco'] = df_final['score_risco'].round(1)

# Normalizando resultado
min_r, max_r = df_final['score_risco'].min(), df_final['score_risco'].max()
df_final['seguranca_normalizada'] = 100 - (((df_final['score_risco'] - min_r) / (max_r - min_r)) * 100)

# Classificação e designação das cores por classe
bins = [-1, 20, 40, 60, 80, 101]
labels = ['Perigoso', 'Segurança crítica', 'Segurança baixa', 'Segurança moderada', 'Segurança alta']
df_final['categoria_seguranca'] = pd.cut(df_final['seguranca_normalizada'], bins=bins, labels=labels)

mapa_cores = {'Segurança alta': 'verde', 'Segurança moderada': 'verde claro', 'Segurança baixa': 'amarelo', 'Segurança crítica': 'laranja', 'Perigoso': 'vermelho'}
df_final['cor'] = df_final['categoria_seguranca'].map(mapa_cores)

df_final = df_final.drop(columns=['seguranca_normalizada'])
df_final = df_final.drop(columns=['safety_score_final_v2'])
df_final = df_final.rename(columns={'risco_infraestrutura': 'peso_infra'})

#

# >>> Verificando saídas/tabelas resultantes

print("\nTabela df_vias tratada: Informações sobre vias")
print(len(df_vias))
display(df_vias.head(3))

print("\nTabela df_vias_score tratada: Score de segurança por vias")
print(len(df_vias_score))
display(df_vias_score.head(3))

print("\nTabela df_seguranca tratada: Roubos e furtos de bicicleta por bairro")
print(len(df_seguranca))
display(df_seguranca.head(3))

print("Tabela df_acidentes tratada: Acidentes por logradouro/vias")
print(len(df_acidentes))
display(df_acidentes.head(3))

print("\nTabela df_logradouros tratada: Velocidade máxima e hierarquia por logradouros e bairro")
print(len(df_logradouros))
display(df_logradouros.head(3))

print("\nTabela df_final: Score de risco por vias")
print(len(df_final))
display(df_final.head(3))


# >>> Downloads

df_acidentes.to_csv('acidentes.csv', index=False)
#files.download('acidentes.csv')

df_seguranca.to_csv('seguranca.csv', index=False)
#files.download('seguranca.csv')

df_logradouros.to_csv('logradouros.csv', index=False, decimal=',')
#files.download('logradouros.csv')

df_vias_score.to_csv('vias_score.csv', index=False)
#files.download('vias_score.csv')

df_vias.to_csv('vias.csv', index=False)
#files.download('vias.csv')

df_final.to_csv('final.csv', index=False, decimal=',')
files.download('final.csv')





# to do:

# usar arvore de decisao (ML) sobre as variaveis para decidir classificacao do risco
# usar scikit learn numpy
# xgb
# kmeans pra agrupar vias em cada tipo de classificacao de risco
# fazer diagrama do fluxo Dados -> Limpeza -> Merge -> Score -> Normalização -> Classificação -> Mapa



Tabela df_vias tratada: Informações sobre vias
576


,infra_tipo,infra_fluxo,vias,inicio,final,bairro,extensao_metros,area_planejamento
0,Faixa compartilhada na calçada,Bidirecional,RODOVIA GOVERNADOR MÁRIO COVAS,Avenida Padre Guilherme Decaminada (Em frente ...,Avenida Padre Guilherme Decaminada (altura da ...,SANTA CRUZ,998.538921,5
1,Faixa compartilhada na calçada,Bidirecional,RUA GENERAL ALEXANDRE BARRETO,Estrada Visconde de Sinimbu,Número 179,SANTA CRUZ,333.073322,5
2,Faixa compartilhada na calçada,Bidirecional,PRAÇA GUILHERME DA SILVEIRA,Em torno da Praça Guilherme da Silveira,Em torno da Praça Guilherme da Silveira,BANGU,554.249452,5



Tabela df_vias_score tratada: Score de segurança por vias
576


,fid,ext,tipo_infra,fluxo_infr,inicio,contabilizar,vias,area_plane,local_infr,final,SHAPE__Length,rp,status,cod_rp,score_infra,cluster_seguranca,score_seguranca_final,safety_score_refinado,safety_score_final_v2
0,1,0.998709,FAIXA COMPARTILHADA NA CALÇADA,BIDIRECIONAL,Avenida Padre Guilherme Decaminada (Em frente ...,Sim,RODOVIA GOVERNADOR MÁRIO COVAS,5,No bordo direito da pista,Avenida Padre Guilherme Decaminada (altura da ...,998.538921,SANTA CRUZ,IMPLANTADA,5.3,4,2,2.5,1.0,0.8
1,2,0.333128,FAIXA COMPARTILHADA NA CALÇADA,BIDIRECIONAL,Estrada Visconde de Sinimbu,Sim,RUA GENERAL ALEXANDRE BARRETO,5,No bordo direito da pista,Número 179,333.073322,SANTA CRUZ,IMPLANTADA,5.3,4,2,2.5,1.0,0.8
2,3,0.554299,FAIXA COMPARTILHADA NA CALÇADA,BIDIRECIONAL,Em torno da Praça Guilherme da Silveira,Sim,PRAÇA GUILHERME DA SILVEIRA,5,Na calçada,Em torno da Praça Guilherme da Silveira,554.249452,BANGU,IMPLANTADA,5.1,4,2,2.5,2.5,2.0



Tabela df_seguranca tratada: Roubos e furtos de bicicleta por bairro
155


...

,logradouro,bairro,hierarquia,velocidade_maxima,peso_h,peso_v,logradouro_padrao
121599,AVENIDA CARLOS LACERDA,ABOLIÇÃO,Estrutural,100,3.0,3.0,AV CARLOS LACERDA
47176,AVENIDA DOM HELDER CAMARA,ABOLIÇÃO,Arterial primária,60,2.5,2.0,AV DOM HELDER CAMARA
46234,BECO DO LOPES,ABOLIÇÃO,Local,20,1.0,1.0,BECO LOPES



Tabela df_final: Score de risco por vias
576


,infra_tipo,infra_fluxo,vias,inicio,final,bairro,extensao_metros,area_planejamento,score_c,score_d,hierarquia,velocidade_maxima,peso_v,peso_h,peso_infra,score_risco,categoria_seguranca,cor
0,Faixa compartilhada na calçada,Bidirecional,RODOVIA GOVERNADOR MÁRIO COVAS,Avenida Padre Guilherme Decaminada (Em frente ...,Avenida Padre Guilherme Decaminada (altura da ...,SANTA CRUZ,998.538921,5,77.6,112.6,NaN,NaN,1.0,1.0,9.2,199.4,Segurança alta,verde
1,Faixa compartilhada na calçada,Bidirecional,RUA GENERAL ALEXANDRE BARRETO,Estrada Visconde de Sinimbu,Número 179,SANTA CRUZ,333.073322,5,77.6,112.6,NaN,NaN,1.0,1.0,9.2,199.4,Segurança alta,verde
2,Faixa compartilhada na calçada,Bidirecional,PRAÇA GUILHERME DA SILVEIRA,Em torno da Praça Guilherme da Silveira,Em torno da Praça Guilherme da Silveira,BANGU,554.249452,5,65.6,119.9,NaN,NaN,1.0,1.0,8.0,193.5,Segurança alta,verde


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>